# Endpoints Flooding Attack Reproduction

## 1. Mechanism overview

### 1.1 Attacking strategy: kube-apiserver as proxy to kube-proxy

This attack implements strategy ❹ in Fig. 3 of the paper, described in §3.7:

> "a malicious container can exploit the kube-apiserver to send a huge number of messages to the kube-proxy on all other nodes for attacking. This can be split into two stages. First, the malicious container sends requests to the kube-apiserver in some way. After that, the treatment logic of this request asks the kube-apiserver to send messages to the kube-proxy on other worker nodes."

Table 1 of the paper lists the specific interfaces this relies on for the second stage: Kube-proxy, `services`/`endpoints`, `watch`, "Watch service/endpoints changes". The mechanism is explained as:

> "the kube-proxy has watch verbs on services and endpoints interfaces [...] It watches the services/endpoints status changes sent by the kube-apiserver, and then modifies the items of the worker node iptables according to the received status change messages [...] a malicious container can use the third attacking strategy (❸ in Fig. 3) to send the pod's status changes to the kube-apiserver. After that, to handle the incoming requests, the kube-apiserver sends related status change messages [...] to the kube-proxy of all the other worker nodes, and the kube-proxy flushes each node's iptables."

This means strategy ❹ reuses strategy ❸ (the readiness probe mechanism from attack3) for its first stage, then adds a second stage of fan-out to every worker node's kube-proxy.

### 1.2 Case 4: endpoints flooding attack (§4.5, Fig. 7)

> "The fourth case exploits the endpoints status changes. In the attack, a malicious container leverages the kube-apiserver as a proxy, and sends endpoints changes to kube-proxy of all nodes in the cluster (❹ in Fig. 3). As a result, it consumes up to 60% CPU resources of the control plane, and flushes all the node's iptables."

Attack steps (§4.5): "This attack has two steps [...] First, the malicious container sends a large number of requests to the kube-apiserver. Then, the incoming requests make the kube-apiserver components send requests to kube-proxy of all other nodes components. Here, we use the readiness probe described in Section 4.4 to achieve the first step."

For the second step, the paper uses the pod to endpoint to service data dependency:

> "a service is an abstract data structure which defines a logical set of pods and an IP address by which to access them, while an endpoint is another structure that binds a set of pods to a specific service [...] Thus, a malicious container can modify its pods status, and leverage the kube-apiserver as a proxy for sending related services/endpoints status changes to the kube-proxy on the worker nodes. After that, the kube-proxy flushes its local iptables. This data dependency [...] makes the control plane forward workloads to all other nodes in the cluster."

### 1.3 What the paper reports as impact

For the local environment, the paper reports (§4.5):

> "Our experiment deploys one hundred malicious containers in one hundred pods on one worker node. Each container leverages the readinessProbe to modify its pod's ready status per second. The pods status changes invoke the kube-apiserver to send related service/endpoints status information to all other nodes' kube-proxy, which flushes all the nodes' iptables. [...] In the local environment, it consumes control plane 60% additional CPU resources, and the kube-proxy flushes all worker nodes' iptables, which is a flooding attack."

Table 3 summarizes this as: "Endpoints flooding attack, strategy ❹, consuming control plane 60% CPU; flushing all nodes' iptables." This is the reference figure we compare our own measurements against in section 3.

## 2. Reproduction: manifests and methodology

### 2.1 Attack topology

The first step reuses the same readiness-probe mechanism as attack3 (§4.4): 100 malicious containers, each running in its own pod, managed by its own separate ReplicaSet and Deployment, pinned to `k8s-w1` via `nodeName`, matching the paper's "one hundred malicious containers in one hundred pods on one worker node" (§4.5).

What attack3 did not need, and this attack does, is a second ingredient for step two: a `Service` object selecting all 100 pods. Each pod carries, alongside its own unique label (needed so each Deployment/ReplicaSet stays independent, per the topology constraint inherited from attack3), a second label shared by all 100 pods. A single `Service` selects on this shared label, so that every readiness flip on any of the 100 pods changes the `Service`'s `Endpoints`/`EndpointSlice` object, which is what kube-proxy watches on every node (Table 1, `endpoints`, `watch`). Without this `Service`, the readiness flips would only reach etcd via the pod/replicaset/deployment chain (attack3's mechanism), not kube-proxy.

### 2.2 Final manifest

Each Deployment runs a single-replica pod whose container's readiness probe is an `exec` check testing the parity of the current Unix timestamp (`test $(( $(date +%s) % 2 )) -eq 0`), run every second (`periodSeconds: 1`, `failureThreshold: 1`, `successThreshold: 1`), identical to attack3's mechanism. The container itself does nothing but idle (`busybox:1.36`, `sleep infinity`); the readiness probe is the only active mechanism.

Each Deployment is generated with its own name (`endpoints-flood-${i}`), its own pod template, and its own `matchLabels` selector, so applying the manifest creates 100 independent Deployment/ReplicaSet/Pod chains. In addition, every pod carries a shared label (`group: endpoints-flood-target`), and a single `Service` (`endpoints-flood-svc`) selects on it, giving all 100 pods one common `Endpoints` object. All pods run under the same restricted security context as the other attacks (non-root, no privilege escalation, all capabilities dropped, `RuntimeDefault` seccomp profile), consistent with the `restricted` Pod Security Standard enforced on `attacker-ns`.

The manifest is produced by `generate-endpoints-flood.sh`, which templates this Deployment definition `COUNT` times (default 100) into a single file, `endpoints-flood-generated.yaml`, followed by the one `Service` definition.

### 2.3 Measurement scripts

**`launch-attack.sh`** applies the generated manifest and waits for the malicious pods to come up:

```bash
kubectl apply -f "$MANIFEST"
kubectl wait --for=condition=Ready pod -n "$NAMESPACE" -l "$LABEL" --timeout=120s || true
```

Cleanup is done with `kubectl delete deployment,service -n attacker-ns -l attack=endpoints-flood`.

**`capture-cpu-loop-attack4.sh`** measures CPU usage of the kube-apiserver process specifically, run on k8s-cp, the same `/proc/<pid>/stat` delta method used for the other attacks (reasoning behind this choice of process in section 3).

**`capture-iptables-snapshot.sh`** snapshots the local node's total iptables rule count and the subset of rules referencing the attack's Service (by name, via iptables comments), run separately on each worker node (k8s-w1, k8s-w2), once before the attack and as a burst of several snapshots (5 by default, 10s apart) during it, each appended as a row to `<label>_iptables_<hostname>.csv`. This is independent evidence of the "flushes all the node's iptables" claim (§4.5), not derived from the CPU measurement.


In [1]:
from aggregate_averages_attack4 import (
    find_cpu_loop_files, find_iptables_files,
    load_rows, CPU_COLS, IPTABLES_COLS, NODES, LABELS,
)
import pandas as pd
from IPython.display import display, HTML

def load_cpu(label):
    files = find_cpu_loop_files(label, base_dir="../results")
    dfs = [load_rows(f, CPU_COLS) for f in files]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=CPU_COLS)

def load_iptables(label, node):
    files = find_iptables_files(label, node, base_dir="../results")
    dfs = [load_rows(f, IPTABLES_COLS) for f in files]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=IPTABLES_COLS)

before_cpu = load_cpu("before")
after_cpu = load_cpu("after")

rows = {LABELS[c]: (before_cpu[c].mean(), after_cpu[c].mean()) for c in CPU_COLS}

for node in NODES:
    before_ipt = load_iptables("before", node)
    after_ipt = load_iptables("after", node)
    for c in IPTABLES_COLS:
        rows[f"{LABELS[c]} ({node})"] = (before_ipt[c].mean(), after_ipt[c].mean())

summary = pd.DataFrame(rows, index=["Before", "After"]).T.round(1)

display(HTML("<h2>Metrics comparison before / after attack</h2>"))
display(summary.style.set_properties(**{"font-size": "16pt"}).set_table_styles(
    [{"selector": "th", "props": [("font-size", "16pt")]}]
))

,Before,After
CPU kube-apiserver process (%),8.400000,106.800000
iptables total rules (k8s-w1),228.000000,2731.800000
iptables rules referencing the attack Service (k8s-w1),0.000000,152.600000
iptables total rules (k8s-w2),228.000000,377.400000
iptables rules referencing the attack Service (k8s-w2),0.000000,108.200000


## 3. Results

Measurements from every `before`/`after` run are aggregated and averaged as follows. The table below compares our reproduction against what the paper reports for this attack in its local environment.

| Metric | Paper (§4.5) | Our reproduction |
|---|---|---|
| CPU consumed by kube-apiserver | up to 60% additional | 8.4% before, 106.9% after, a 98.4-point increase |
| iptables flush on worker nodes | described qualitatively as flushing all nodes' iptables, no figure given | total rules on k8s-w1 rose from 228 to an average of 2,731.8 (about 12-fold); on k8s-w2 from 228 to an average of 380.0 (about 1.7-fold); rules referencing the attack's Service rose from 0 to an average of 152.6 (k8s-w1) and 109.3 (k8s-w2) |

Two things stand out from this comparison.

**CPU consumption was measured on kube-apiserver specifically, the only control-plane process the paper names for this attack.** Unlike Cases 2 and 3, the paper's description of Case 4 does not attribute its 60% figure to a named process, only to "the control plane" generically. Since kube-apiserver is the sole control-plane component explicitly described as doing work in the mechanism ("the kube-apiserver components send requests to kube-proxy", §4.5), `capture-cpu-loop-attack4.sh` measures kube-apiserver's process CPU specifically, the same `/proc/<pid>/stat` delta method used for the other attacks. The measured increase (+98.4 points, from 8.4% to 106.9%) is larger than the paper's reported 60%, and crosses 100%, which is possible with a per-process reading on a multi-threaded process but would not be possible under a normalized, all-cores-capped reading. As discussed for the CoreDNS and kube-apiserver attacks, the paper's own Fig. 5 keeps every attack under 100%, which suggests its figures may be normalized in a way our per-process measurement is not; the two numbers likely sit on different scales, which limits how directly they can be compared.

**The iptables flush is confirmed independently of the CPU figures, but with an asymmetry between the two worker nodes that the paper does not address.** On both nodes, the rule count baseline before the attack is identical (228 total rules, 0 referencing the attack's Service), and both show a clear increase once the attack is running, confirming the mechanism described in §4.5 ("the kube-proxy flushes all worker nodes' iptables"). However, the increase is far larger on k8s-w1 (about 12-fold) than on k8s-w2 (about 1.7-fold). The 100 malicious pods are all scheduled on k8s-w1 via `nodeName` (matching the paper's own setup, which also runs its hundred containers "on one worker node", §4.5), so k8s-w1 locally hosts the pods targeted by the Service in addition to receiving the same endpoints-change notifications as every other node; this additional local factor is a plausible but unconfirmed explanation for the gap, not something the paper's text addresses directly. The paper itself reports no numeric figure for the iptables flush in any environment, only the qualitative statement that it occurs, so this comparison has no paper-reported value to validate against; it stands as an independent confirmation of the mechanism rather than a figure to match.